# Глава 7. Обучение с подкреплением

В прошлой главе мы рассмотрели второй этап обучения больших языковых моделей - донастройка по размечкенным данным и методы оптимизации данного процесса.

В этой главе рассмотрим третий этап обучения - обучение с подкреплением (Reinforcment Learning) и посмотрим, как оно стало главным инструментом доводки (alignment) языковых моделей в соотвествии с пользовательскими предпочтениями. Начнем с обзора базовых алгоритмов RL, рассмотрим первые примеры применения к языковым моделям RLHF и DPO, и закончим использованием в рассуждающих моделях уровня o1 и DeepSeek-R1.

## Мотивация
Предобучение через предсказание следующего токена учит модель правдоподобно продолжать текст. Supervised fine-tuning (SFT) сдвигает распределение так чтобы ответ получался ближе к формату «инструкция — ответ», но по своей природе остаётся имитацией: модель воспроизводит показанные демонстрации и не получает никакого сигнала о том, чем хороший ответ отличается от посредственного. 

Между целью обучения (правдоподобие) и целью использования (полезность, безопасность, правдивость) остаётся зазор, и закрыть его «правильной функцией потерь» не выходит: такие свойства, как полезность или честность, мы попросту не умеем записывать в виде дифференцируемого лосса по токенам.

Обучение с подкреплением предлагает обходной путь. Если цель нельзя задать лоссом, её можно задать наградой — числом, сообщающим модели, насколько удачным оказался уже готовый ответ. Кто и как выставляет это число (человек, другая модель, автоматическая проверка) — и определяет облик конкретного метода.

## Обзор классического RL

### Постановка задачи

RL описывает агента, который взаимодействует со средой во времени. Стандартный формализм — марковский процесс принятия решений (MDP): на каждом шаге агент наблюдает состояние $s_t$, выбирает действие $a_t$ согласно своей политике $\pi(a \mid s)$ — распределению действий при данном состоянии, — получает награду $r_t$, после чего среда переходит в новое состояние по закону $P(s_{t+1} \mid s_t, a_t)$. Цель — найти политику, максимизирующую ожидаемую суммарную дисконтированную награду

$$J(\pi) = \mathbb{E}_{\pi}\Big[\sum_{t} \gamma^t r_t\Big],$$

где $\gamma \in [0,1)$ задаёт, насколько агент ценит будущее по сравнению с настоящим. Принципиальное отличие от обучения с учителем: правильных действий агенту никто не показывает, о качестве решений он узнаёт только через награду — часто отложенную и редкую. Отсюда две вечные трудности RL: баланс исследования и эксплуатации (нужно пробовать новое, чтобы находить лучшее) и credit assignment (какое из множества действий в эпизоде на самом деле привело к успеху или провалу).

### Ценностные функции и advantage

Введем два ключевых термина: 
- функция $V^\pi(s)$ — ожидаемый суммарный выигрыш, если стартовать из состояния $s$ и дальше действовать по политике $\pi$;
- функция $Q^\pi(s,a)$ — то же самое при условии, что первым действием будет $a$. Их разность

$$A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$$

называется advantage и отвечает на вопрос «насколько конкретное действие $a$ лучше среднего поведения политики в этом состоянии»

### Два пути к политике

Исторически сложились два семейства методов. В рамках Value-based подхода мы моделируем функцию ценности, а политика опредееляется жадно по принципу «всегда выбирай действие с максимальным выигрышем $Q$»; К этой группе относятся алгоритм **Q-learning** и его нейросетевая версия **DQN** [(Mnih et al., 2013)](https://arxiv.org/abs/1312.5602), прославившаяся играми Atari. 

В Policy-based методах выбор следующего действия параметризуется функцией, соотвественно на обучении мы можем корреткировать её методом градиентного спуска относительно ожидаемой награды $\nabla_\theta J$.

Есть и гибридная схема actor-critic, которая совмещает в себе два компонента. Первый компонент «актор» реализует политику - выбирает действие. Второй компонент «критик» моделирует ценность и его роль - считать advantage и уменьшать шум градиента

Для языковых моделей практичным оказался именно policy-градиентный путь. В конектсте языковых моделей пространство действий — это словарь на десятки тысяч токенов, выбор действия дискретен и требует перебора, из-за чего считать по нему значения $Q$ неудобно. Зато функция политики уже существует по построению: предобученная языковая модель это и есть распределение над следующим токеном

### Policy gradient и REINFORCE

Теорема о градиенте политики даёт выражение

$$\nabla_\theta J = \mathbb{E}\big[\nabla_\theta \log \pi_\theta(a \mid s)\cdot A(s,a)\big],$$

которое читается просто: повышай вероятность действий, оказавшихся лучше среднего, и понижай для остальных; сила сдвига пропорциональна advantage. Простейшая реализация этой идеи — алгоритм **REINFORCE** (Williams, 1992): сэмплируем траектории целиком и вместо advantage подставляем фактически полученную награду (за вычетом базовой линии). Метод работает, но страдает от огромной дисперсии градиента: одна удачная или неудачная траектория может резко дёрнуть параметры, и обучение легко расходится.

### PPO: policy gradient с ремнём безопасности

Проблема резких обновлений в RL особенно болезненна, потому что испорченная политика портит и данные, которые сама же собирает, — ошибка накапливается. **PPO** [(Schulman et al., 2017)](https://arxiv.org/abs/1707.06347) предлагает решение: ограничить, насколько новая политика может отличаться от старой за одно обновление. Формально это clipped-объектив

$$L^{\text{CLIP}} = \mathbb{E}\Big[\min\big(r_t(\theta) A_t,\; \text{clip}(r_t(\theta),\, 1-\epsilon,\, 1+\epsilon)\, A_t\big)\Big], \qquad r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}.$$

Как только отношение вероятностей новой и старой политики выходит за коридор $[1-\epsilon,\, 1+\epsilon]$, градиент по нему обнуляется: выгодные, но слишком резкие шаги отсекаются. Запоминать стоит не формулу, а идею: PPO — это policy gradient, которому запрещены резкие движения. Именно благодаря устойчивости он стал алгоритмом по умолчанию в глубоком RL — и в этой роли пришёл в языковые модели.

## Генерация текста как задача RL

Прежде чем двигаться дальше, зафиксируем словарь соответствий, на котором держится всё последующее. Промпт — начальное состояние. Действие — выбор очередного токена из словаря. Состояние после шага — конкатенация промпта и уже сгенерированных токенов. Политика — сама языковая модель: она и есть $\pi_\theta(a \mid s)$, распределение над следующим токеном при данном префиксе. Эпизод — генерация одного ответа до токена конца последовательности; законченный ответ — это траектория.

Получившийся MDP устроен своеобразно. Переходы детерминированы: новое состояние — это просто старое плюс выбранный токен, никакой внешней динамики среды нет. Горизонт длинный — сотни и тысячи шагов. А награда почти всегда терминальная: оценить можно только законченный ответ, отдельный токен сам по себе не «хорош» и не «плох». Отсюда обострённая проблема credit assignment: если ответ из тысячи токенов получил низкую оценку, какие именно решения были ошибочными? Наконец, есть роскошь, которой в классическом RL нет: стартовая политика не случайна, а уже очень хороша — это предобученная модель. Поэтому задача звучит не как «выучить поведение с нуля», а как «аккуратно сдвинуть готовое распределение», и почти каждый метод ниже содержит тот или иной якорь, не позволяющий уйти далеко от исходной модели.

## Обучение на человеческих предпочтениях

### Сравнивать проще, чем описывать

Итак, нужен источник награды. Прямой путь — просить людей ставить ответам абсолютные оценки — работает плохо: такие оценки шумны и слабо согласуются между разметчиками. Это наблюдение сделали ещё до эпохи LLM [(Christiano et al., 2017)](https://arxiv.org/abs/1706.03741): людям гораздо легче сравнить два варианта, чем оценить один, а по накопленным попарным сравнениям можно обучить модель награды и уже под неё оптимизировать политику обычным RL. Идею сначала проверили на играх и робототехнике, затем перенесли на текст — на задачу суммаризации [(Stiennon et al., 2020)](https://arxiv.org/abs/2009.01325).

### RLHF: канонический пайплайн

В полную силу схема заработала в модели InstructGPT [(Ouyang et al., 2022)](https://arxiv.org/abs/2203.02155) — работе, которая легла в основу ChatGPT и закрепила за подходом имя **RLHF** (reinforcement learning from human feedback). Пайплайн InstructGPT состоит из трёх этапов.

Сначала идет supervised fine-tuning: базовую модель дообучают на демонстрациях «инструкция — качественный ответ» и получают разумную стартовую политику $\pi_{\text{SFT}}$.

Затем обучают модель награды (reward model, RM). На каждый промпт генерируют несколько ответов, разметчики их ранжируют, и отдельная модель учится предсказывать человеческое предпочтение одним скаляром — обычно через лосс Брэдли–Терри

$$L_{\text{RM}} = -\log \sigma\big(r(x, y_w) - r(x, y_l)\big),$$

где $y_w$ — предпочтённый ответ, а $y_l$ — отвергнутый. Смысл этой стадии: RM сжимает дорогие человеческие суждения в дешёвую функцию, которую можно вызывать миллионы раз.

Наконец, RL-стадия: политику оптимизируют с помощью PPO, максимизируя награду RM, но с принципиальной поправкой — штрафом за отклонение от SFT-модели:

$$\text{reward} = r_{\text{RM}}(x,y) - \beta\,\mathrm{KL}\big[\pi_\theta(\cdot \mid x)\,\|\,\pi_{\text{SFT}}(\cdot \mid x)\big].$$

KL-слагаемое — тот самый якорь из предыдущего раздела: без него политика быстро находит вырожденные тексты, которым несовершенная RM ошибочно ставит высокий балл, и перестаёт быть осмысленной языковой моделью.

```
[Base LM] --SFT--> [SFT model] --(сбор сравнений)--> [Reward Model]
                        |                                  |
                        +------------- PPO ----------------+
                                        |
                                 [Aligned model]
                        (награда = RM − β·KL до SFT)
```

Результат хорошо известен: модели научились следовать инструкциям, выдерживать тон, отказываться от вредных запросов, и сравнительно дешёвая RLHF-доводка дала пользователю больше, чем очередное масштабирование претрена. Но проявилась и цена. Пайплайн требует одновременно держать три-четыре модели — политику, замороженную reference-копию для KL, reward model и критика для PPO; сам PPO на тексте капризен к гиперпараметрам. А главное — обнажилась проблема, которая станет сквозной для всей области.

### Reward hacking и закон Гудхарта

Политика в RLHF оптимизирует не «качество ответа», а его прокси — балл reward model, обученной на конечном наборе сравнений. Когда оптимизационное давление становится достаточно сильным, срабатывает закон Гудхарта: когда мера становится целью, она перестаёт быть хорошей мерой. Модель начинает эксплуатировать слабости судьи вместо того, чтобы улучшать ответы. Типичные проявления хорошо задокументированы: раздувание длины (RM статистически предпочитают развёрнутые ответы — и модель учится лить воду), sycophancy — привычка соглашаться с пользователем вместо того, чтобы его поправить, уверенный тон вместо точности, эффектное оформление вместо содержания.

Полностью проблема не решается: пока награда — прокси, её можно взламывать. Её сдерживают несколькими способами: KL-якорь ограничивает, как далеко политика может уйти в поисках лазеек; reward model периодически переобучают на свежих сравнениях, где текущая политика уже пыталась схитрить; используют ансамбли судей. Полезно держать reward hacking в голове как красную нить главы: почти каждый следующий метод можно читать как попытку сделать награду честнее или менее взламываемой.

### Обратная связь от модели: Constitutional AI и RLAIF

Второе узкое место RLHF — сами люди. Разметка предпочтений дорога, медленна и плохо масштабируется, а по мере усложнения ответов разметчикам всё труднее их сравнивать. **Constitutional AI** [(Bai et al., 2022)](https://arxiv.org/abs/2212.08073) предложил заменить разметчика моделью, управляемой явным набором принципов — «конституцией». Сначала модель сама критикует и переписывает собственные ответы в соответствии с принципами, и на этих правках делается SFT; затем модель-судья, опираясь на конституцию, размечает пары предпочтений, на которых обучается reward model и запускается стандартная RL-стадия. Обучение по AI-разметке получило собственное имя — **RLAIF** [(Lee et al., 2023)](https://arxiv.org/abs/2309.00267) — и эмпирически даёт качество, сопоставимое с человеческой разметкой, при кратно меньшей стоимости. Человеческий труд при этом смещается на уровень выше: вместо миллионов попарных сравнений люди пишут десятки принципов. Проблема прокси, впрочем, не исчезает — она переезжает в модель-судью вместе с её смещениями.

## Проще: предпочтения без RL-машинерии

Раз RLHF-пайплайн так тяжёл, естественно спросить: нельзя ли получить тот же эффект без RL вовсе? **DPO** [(Rafailov et al., 2023)](https://arxiv.org/abs/2305.18290) отвечает: можно. Авторы заметили, что задача «максимизируй награду при KL-ограничении» имеет решение в замкнутой форме, связывающее оптимальную политику с наградой, — и, подставив эту связь в лосс Брэдли–Терри, награду можно исключить из уравнений совсем. Остаётся простой лосс непосредственно по парам предпочтений:

$$L_{\text{DPO}} = -\log \sigma\Big(\beta \log\frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log\frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\Big).$$

Интуитивно DPO повышает вероятность предпочтённых ответов и понижает вероятность отвергнутых, измеряя и то и другое относительно reference-модели — всё тот же якорь, только встроенный прямо в лосс. Ни reward model, ни сэмплирования во время обучения, ни PPO: выравнивание превращается в обучение с учителем на парах — стабильное, дешёвое и воспроизводимое, за что метод мгновенно стал стандартом в открытых моделях.

Цена удобства — офлайновость. DPO учится на фиксированном датасете и не порождает новых ответов в процессе; PPO-подобные методы, напротив, всё время исследуют свежие сэмплы текущей политики, что при качественной награде даёт более высокий потолок. Этот размен — простота офлайн-обучения против потенциала онлайн-исследования — до сих пор определяет выбор между семействами. Вокруг DPO выросла целая экосистема модификаций: **IPO** [(Azar et al., 2023)](https://arxiv.org/abs/2310.12036) борется с переобучением на предпочтениях, **KTO** [(Ethayarajh et al., 2024)](https://arxiv.org/abs/2402.01306) обходится отдельными метками «хорошо/плохо» без пар, **ORPO** [(Hong et al., 2024)](https://arxiv.org/abs/2403.07687) сливает SFT и выравнивание в одну стадию. Общий вектор один: меньше движущихся частей.

## От вкуса к истине: проверяемые награды

До 2024 года RL для LLM почти всегда означал выравнивание под предпочтения — вкус, тон, безопасность. Затем центр тяжести сместился на задачи рассуждения: математику, программирование, логику. У этих задач есть решающее свойство — правильность ответа проверяется автоматически: число сверяется с эталоном, код прогоняется через тесты, формат — через регулярное выражение. Награду больше не нужно предсказывать обученной моделью, её можно вычислить. Подход получил имя **RLVR** (reinforcement learning with verifiable rewards); термин закрепила работа Tülu 3 [(Lambert et al., 2024)](https://arxiv.org/abs/2411.15124). Такая награда дёшева, масштабируема и — главное — куда устойчивее к взлому: обмануть unit-тесты труднее, чем модель-судью. Плата — узость применения: нужен проверяемый правильный ответ, и сигнал остаётся редким, терминальным.

| | Награда из предпочтений (RLHF/DPO) | Проверяемая награда (RLVR) |
|---|---|---|
| Источник | модель-судья на человеческих сравнениях | правило: сверка ответа, тесты, формат |
| Природа сигнала | субъективный, шумный, взламываемый | объективный, дешёвый, масштабируемый |
| Область применения | стиль, безопасность, любые задачи | задачи с проверяемым ответом |
| Главный риск | reward hacking, sycophancy | редкий сигнал только в конце эпизода |

### GRPO: онлайн-RL без критика

Оставалась инженерная тяжесть самого PPO: для оценки advantage ему нужен критик — отдельная сеть размера, сопоставимого с политикой, что почти удваивает память и вычисления; к тому же обучить хороший критик при чисто терминальной награде само по себе трудно. **GRPO** [(Shao et al., 2024)](https://arxiv.org/abs/2402.03300), предложенный в работе DeepSeekMath, убирает критик одним изящным приёмом. На каждый промпт сэмплируется группа из $G$ ответов; каждый получает награду $r_i$ (например, от верификатора); advantage ответа вычисляется как его отклонение от среднего по группе:

$$A_i = \frac{r_i - \text{mean}(r_1,\dots,r_G)}{\text{std}(r_1,\dots,r_G)}.$$

Точкой отсчёта служит не обученная функция ценности, а «одноклассники» — другие ответы модели на тот же вопрос. Дальше применяется тот же clipped-объектив в духе PPO. Результат: полноценный онлайн-RL примерно вдвое дешевле и без хрупкого компонента; именно эта экономия сделала возможными масштабные RL-эксперименты следующего шага.

### DeepSeek-R1: рассуждение из чистого RL

Осенью 2024 года OpenAI показала [o1](https://openai.com/index/learning-to-reason-with-llms/) — первую «рассуждающую» модель, обученную с помощью RL генерировать длинные цепочки мыслей перед ответом, — но рецепт не раскрыла. Открытым ответом стала **DeepSeek-R1** [(DeepSeek-AI, 2025)](https://arxiv.org/abs/2501.12948), и её эксперимент R1-Zero — вероятно, самый обсуждаемый результат года. Авторы взяли базовую модель и запустили GRPO вообще без SFT-стадии, с наградой лишь за два проверяемых свойства: правильность финального ответа и соблюдение формата рассуждения. Гипотеза состояла в том, что человеческие демонстрации могут ограничивать поиск, а свободная RL-оптимизация позволит модели самой найти удачные стратегии. Так и вышло: по мере обучения модель спонтанно удлиняла цепочки рассуждений, научилась перепроверять себя, замечать ошибку, возвращаться и пробовать другой путь — этим паттернам её никто не учил, они возникли под давлением сигнала «ответ верен».

У чистого RL нашлись практические дефекты — нечитаемые рассуждения, смешение языков, — поэтому финальная R1 обучена многостадийным конвейером, чередующим парадигмы: короткий cold-start SFT задаёт чистый формат рассуждения; GRPO с проверяемыми наградами наращивает качество; из полученной модели rejection sampling отбирает лучшие траектории для нового SFT (уже с примесью обычных, не-reasoning данных); финальная RL-стадия добавляет выравнивание под предпочтения. Обратите внимание, как размылась граница: SFT и RL здесь — не «стадия два и стадия три», а инструменты, которые чередуют столько раз, сколько нужно.

Концептуальный сдвиг стоит проговорить отдельно. RLHF использовал RL, чтобы придать модели вкус и манеры, — способности при этом оставались прежними. R1 показала, что RL может выращивать новую способность — длинное многошаговое рассуждение — из одного лишь бинарного сигнала правильности.

### Новая ось масштабирования: вычисления на инференсе

У reasoning-моделей есть следствие, выходящее за рамки собственно RL. Классический рецепт улучшения LLM — больше данных и параметров на этапе обучения. Модели типа o1 и R1 добавили вторую ось: качество растёт, если дать модели больше вычислений в момент ответа, то есть позволить «думать дольше». RL здесь играет роль учителя, который делает долгое размышление продуктивным: именно под наградой за правильность модель осваивает перебор подходов, самопроверку и откаты — и дополнительные токены превращаются в дополнительную точность, а не в воду. Так оформился test-time scaling — отдельная парадигма масштабирования, тесно переплетённая с RL-обучением; к ней стоит вернуться, если в курсе будет тема про законы масштабирования.

### Уточнения и открытые проблемы

Стандартный GRPO небезупречен: обучение подвержено коллапсу энтропии (модель слишком рано теряет разнообразие ответов), а нормировки внутри группы создают смещения — в частности, систематически раздувают длину ответов. **Dr. GRPO** [(Liu et al., 2025)](https://arxiv.org/abs/2503.20783) устраняет эти смещения, убирая проблемные нормировки; **GSPO** [(Zheng et al., 2025)](https://arxiv.org/abs/2507.18071) переносит importance-отношения с отдельных токенов на последовательность целиком, что резко стабилизирует обучение больших MoE-моделей (алгоритм лёг в основу RL-обучения Qwen3). Против редкости терминальной награды развивают процессные модели награды (process reward models) — судей, оценивающих каждый шаг рассуждения, а не только итог; работа [(Lightman et al., 2023)](https://arxiv.org/abs/2305.20050) показала, что пошаговый контроль надёжнее итогового. Наконец, RL двинулся в агентные сценарии, где модель по ходу рассуждения вызывает инструменты — поиск, интерпретатор кода — а награда выдаётся за успех всей многошаговой задачи; характерный пример — **Search-R1** [(Jin et al., 2025)](https://arxiv.org/abs/2503.09516), где модель учится перемежать рассуждение с поисковыми запросами.
